In [32]:
import time
import hashlib
import random

In [33]:

class Block:
    def __init__(self, index, previous_hash, data, timestamp=None):
        self.index = index
        self.previous_hash = previous_hash
        self.timestamp = timestamp or time.time()
        self.data = data
        self.nonce = 0
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_contents = f"{self.index}{self.previous_hash}{self.timestamp}{self.data}{self.nonce}"
        return hashlib.sha256(block_contents.encode()).hexdigest()

    def __repr__(self):
        return (f"\n--- Block {self.index} ---\n"
                f"Hash        : {self.hash}\n"
                f"Prev Hash   : {self.previous_hash}\n"
                f"Timestamp   : {self.timestamp}\n"
                f"Nonce       : {self.nonce}\n"
                f"Data        : {self.data}\n")


class Blockchain:
    def __init__(self, difficulty=2, mining_reward=50):
        self.chain = [self.create_genesis_block()]
        self.difficulty = difficulty
        self.mining_reward = mining_reward
        self.balances = {}  # store miner rewards

    def create_genesis_block(self):
        return Block(0, "0", "Genesis Block", time.time())

    def get_last_block(self):
        return self.chain[-1]

    def proof_of_work(self, block):
        target_prefix = "0" * self.difficulty
        while not block.hash.startswith(target_prefix):
            block.nonce += 1
            block.hash = block.calculate_hash()
        return block.hash

    def add_block(self, data, miner):
        last_block = self.get_last_block()
        new_block = Block(last_block.index + 1, last_block.hash, data)

        print(f"\n⛏️ Mining block {new_block.index} by {miner}...")
        self.proof_of_work(new_block)

        # Add block to chain
        self.chain.append(new_block)

        # Reward the miner
        self.balances[miner] = self.balances.get(miner, 0) + self.mining_reward

        # Print full block details
        print("✅ Block mined successfully!")
        print(new_block)

        print(f"💰 Reward: {self.mining_reward} coins awarded to {miner}")
        print(f"🏦 {miner} Balance: {self.balances[miner]} coins\n")



In [34]:
# Define the PoSBlock class which inherits from Block and includes a validator attribute
class PoSBlock(Block):
    def __init__(self, index, previous_hash, data, validator, timestamp=None):
        """
        Initialize a new block for the PoS blockchain.

        Args:
            index (int): Block position in the chain.
            previous_hash (str): Hash of the previous block.
            data (str): Data stored in the block.
            validator (str): The selected validator for this block.
            timestamp (float): Creation time of the block.
        """
        super().__init__(index, previous_hash, data, timestamp)
        self.validator = validator  # Validator who approved or created the block

    def __repr__(self):
        """
        Return a string representation of the PoS block.
        """
        return (f"PoSBlock(Index: {self.index}, Hash: {self.hash[:10]}..., "
                f"Prev_Hash: {self.previous_hash[:10]}..., Validator: {self.validator}, Data: {self.data})")


# Define the PoSBlockchain class for the Proof of Stake protocol
class PoSBlockchain:
    def __init__(self, validators):
        """
        Initialize the PoS blockchain with a genesis block and a list of validators.

        Args:
            validators (dict): A dictionary of validators and their corresponding stakes,
                               e.g., {"Alice": 50, "Bob": 30, "Charlie": 20}.
        """
        self.chain = [self.create_genesis_block()]
        self.validators = validators

    def create_genesis_block(self):
        """
        Create the genesis block for the PoS blockchain.

        Returns:
            PoSBlock: The genesis block with a dummy validator.
        """
        genesis_block = PoSBlock(0, "0", "Genesis PoS Block", "Genesis", time.time())
        genesis_block.hash = genesis_block.calculate_hash()
        return genesis_block

    def get_last_block(self):
        """
        Retrieve the latest block in the PoS blockchain.

        Returns:
            PoSBlock: The last block in the chain.
        """
        return self.chain[-1]

    def select_validator(self):
        """
        Select a validator based on their stake.

        Higher stake means a higher chance of being chosen. This is a simplified mechanism.

        Returns:
            str: The name of the selected validator.
        """
        validators = list(self.validators.keys())
        stakes = list(self.validators.values())
        selected_validator = random.choices(validators, weights=stakes, k=1)[0]
        return selected_validator

    def add_block(self, data):
        """
        Add a new block to the PoS blockchain.

        A validator is selected based on stake and then the block is created.

        Args:
            data (str): The data to include in the new block.
        """
        last_block = self.get_last_block()
        new_index = last_block.index + 1
        validator = self.select_validator()
        # Retrieve the stake of the selected validator from the dictionary
        stake = self.validators[validator]
        new_block = PoSBlock(new_index, last_block.hash, data, validator)
        # No extensive mining is required in PoS; just calculate the hash
        new_block.hash = new_block.calculate_hash()
        self.chain.append(new_block)
        print(f"Block {new_index} added by validator '{validator}' with stake {stake} and hash: {new_block.hash}\n")

In [35]:
if __name__ == "__main__":
    # Testing Proof of Work (PoW) Blockchain
    print("=== Testing Proof of Work (PoW) Blockchain ===\n")
    # Initialize PoW blockchain with a chosen difficulty level (e.g., 3)
    blockchain = Blockchain(difficulty=3)
    blockchain.add_block("Block 1 Data - PoW", miner="Alice")
    blockchain.add_block("Block 2 Data - PoW", miner="Bob")
    blockchain.add_block("Block 3 Data - PoW", miner="Alice")
    # Display the PoW blockchain
    print("PoW Blockchain:")
    print("\n=== Final Balances ===")
    for miner, balance in blockchain.balances.items():
        print(f"{miner}: {balance} coins")

    # Testing Proof of Stake (PoS) Blockchain
    print("\n=== Testing Proof of Stake (PoS) Blockchain ===\n")
    # Define a dictionary of validators with their stakes
    validators = {
        "Alice": 50,
        "Bob": 30,
        "Charlie": 20
    }
    pos_blockchain = PoSBlockchain(validators)
    pos_blockchain.add_block("Block 1 Data - PoS")
    pos_blockchain.add_block("Block 2 Data - PoS")
    pos_blockchain.add_block("Block 3 Data - PoS")

    # Display the PoS blockchain
    print("PoS Blockchain:")
    for block in pos_blockchain.chain:
        print(block)

=== Testing Proof of Work (PoW) Blockchain ===


⛏️ Mining block 1 by Alice...
✅ Block mined successfully!

--- Block 1 ---
Hash        : 000189b93537c958a5c4accccfe7cad070c5d7a82116eda913dbc6a5fdbd9a91
Prev Hash   : ae3482dfbd6ecc888832292bd18d89ea8a9c395215ab9d1ef06451f66548fb9e
Timestamp   : 1777556326.660567
Nonce       : 14095
Data        : Block 1 Data - PoW

💰 Reward: 50 coins awarded to Alice
🏦 Alice Balance: 50 coins


⛏️ Mining block 2 by Bob...
✅ Block mined successfully!

--- Block 2 ---
Hash        : 00028c04649a02f01c4723f4514eb73bfa038f0894f6b9fc6d368cd78e7a613d
Prev Hash   : 000189b93537c958a5c4accccfe7cad070c5d7a82116eda913dbc6a5fdbd9a91
Timestamp   : 1777556326.7333496
Nonce       : 2242
Data        : Block 2 Data - PoW

💰 Reward: 50 coins awarded to Bob
🏦 Bob Balance: 50 coins


⛏️ Mining block 3 by Alice...
✅ Block mined successfully!

--- Block 3 ---
Hash        : 000681688f00b9eff976b388db4a7ce598ac5dbea69277b1a747a1472430100b
Prev Hash   : 00028c04649a02f01c4723f